# 🔁 LLM Lie Detector: research loop

Claude improves a FIBS lie-detection probe in a loop: **propose → train → evaluate on unseen lie families → keep or revert → repeat**.

1. `Runtime → Change runtime type → GPU` (T4 is fine, L4/A100 faster)
2. Add your Anthropic API key under **🔑 Secrets** (left sidebar) as `ANTHROPIC_API_KEY`, notebook access ON
3. Run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = 'people-looking-at-my-laptop-liked-it'
!rm -rf LLM_lie_detector && git clone -q -b $BRANCH https://github.com/msverma101/LLM_lie_detector.git
%cd LLM_lie_detector/experiments/research_loop
!git log --oneline -1

In [ ]:
# Colab already has torch, pandas, numpy; don't upgrade pandas (google-colab pins it)
!pip install -q -U transformers accelerate anthropic huggingface_hub

## Step 1: cache activations (the only GPU step, ~20 min on a T4)
Resumable: if Colab disconnects, rerun this cell. Run the quick test first.

In [ ]:
# QUICK TEST first (~220 rows, a few minutes): checks the whole pipeline end to end
!python cache_activations.py --model Qwen/Qwen3.5-2B --n_per_family 8 --out acts_test && python eval.py --acts acts_test

In [ ]:
MODEL = 'Qwen/Qwen3.5-2B'   # fallback if too slow: 'Qwen/Qwen3.5-0.8B'
N_PER_FAMILY = 200          # ~5,600 rows, ~20 min on a T4
!python cache_activations.py --model $MODEL --n_per_family $N_PER_FAMILY --out acts

## Step 2: baseline (the FIBS paper's LinearEnsemble probe)

In [ ]:
!python eval.py --probe probe.py --acts acts

## Step 3: the research loop (Claude Opus 5.5 proposes, eval decides)
Each iteration costs roughly $0.2–0.5 in API usage.

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('key loaded:', bool(os.environ['ANTHROPIC_API_KEY']))

In [ ]:
ITERS = 15
!python loop.py --acts acts --iters $ITERS

## Step 4: results

In [ ]:
!python report.py
from IPython.display import Image, Markdown, display
display(Image('runs/progress.png'))
display(Markdown(open('runs/summary.md').read()))

In [ ]:
# download everything (results + all tried probes)
!zip -qr research_loop_results.zip runs
from google.colab import files
files.download('research_loop_results.zip')